In [3]:
from selenium import webdriver
from bs4 import BeautifulSoup
import time
url = "https://cidbregistration.govmu.org/cidbfront/pages/listOfValidCertificates.xhtml"
driver = webdriver.Chrome()
driver.get(url)
time.sleep(3)
soup = BeautifulSoup(driver.page_source,"html")
soup.prettify()

'<html lang="en" xmlns="http://www.w3.org/1999/xhtml">\n <head id="j_idt2">\n  <meta content="IE=edge" http-equiv="X-UA-Compatible"/>\n  <meta content="text/html; " http-equiv="Content-Type"/>\n  <meta content="width=device-width, initial-scale=1.0, maximum-scale=1.0, user-scalable=0" name="viewport"/>\n  <meta content="yes" name="apple-mobile-web-app-capable"/>\n  <link href="/cidbfront/javax.faces.resource/theme.css.xhtml?ln=primefaces-manhattan-teal-yellow" rel="stylesheet" type="text/css"/>\n  <link href="/cidbfront/javax.faces.resource/fa/font-awesome.css.xhtml?ln=primefaces&amp;v=6.2" rel="stylesheet" type="text/css"/>\n  <link href="/cidbfront/javax.faces.resource/components.css.xhtml?ln=primefaces&amp;v=6.2" rel="stylesheet" type="text/css"/>\n  <script src="/cidbfront/javax.faces.resource/jquery/jquery.js.xhtml?ln=primefaces&amp;v=6.2" type="text/javascript">\n  </script>\n  <script src="/cidbfront/javax.faces.resource/jquery/jquery-plugins.js.xhtml?ln=primefaces&amp;v=6.2" ty

In [4]:
table_body = soup.find("tbody" , class_ ="ui-datatable-data ui-widget-content")
table = table_body.find_all("tr" , class_ ="ui-widget-content ui-datatable-even")
table[0]

<tr class="ui-widget-content ui-datatable-even" data-ri="0" role="row"><td class="active-text" role="gridcell">JIANGXI - ARWAN JV</td><td class="active-text" role="gridcell">CIVIL ENGINEERING CONSTRUCTION WORKS</td><td class="active-text" role="gridcell" style="text-align:center;">A</td><td class="active-text" role="gridcell">CT/CE/JV/PF/A/04038</td><td class="active-text" role="gridcell">22 May 2026</td><td class="active-text" role="gridcell">21 May 2027</td><td class="active-text" role="gridcell">VIKRAM DOOKEE</td><td class="active-text" role="gridcell">arwanenterprise@gmail.com</td><td class="active-text" role="gridcell">52542742/2465050</td></tr>

In [ ]:
table[0].find("td" , class_="active-text").text
table[0].find("td" , class_="active-text")

KeyError: 0

In [4]:
import os
import csv
import time
import random
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager

# --- CONFIGURATION ---
TARGET_URL = "https://cidbregistration.govmu.org/cidbfront/pages/listOfValidCertificates.xhtml"
OUTPUT_FILE = "cidb_valid_certificates_clean.csv"
HEADERS = [
    "Company Name", "Class of Work", "Grade", "Certificate No", 
    "Expiry Date", "Effective Date", "Contact Person", "Email", "Phone"
]

def setup_stealth_driver():
    options = webdriver.ChromeOptions()
    options.add_argument("--headless=new")  
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("--disable-blink-features=AutomationControlled")
    options.add_argument("--blink-settings=imagesEnabled=false") 
    options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36")
    
    driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)
    driver.execute_cdp_cmd("Page.addScriptToEvaluateOnNewDocument", {
        "source": "Object.defineProperty(navigator, 'webdriver', {get: () => undefined})"
    })
    driver.set_page_load_timeout(30)
    return driver

def scrape_visible_rows(driver, csv_writer):
    """Uses Selenium instead of BeautifulSoup to target ONLY active table rows."""
    # This CSS selector looks inside the data table body for rows that explicitly have a data-ri index
    visible_rows = driver.find_elements(By.CSS_SELECTOR, "tbody.ui-datatable-data tr[data-ri]")
    
    count = 0
    for row in visible_rows:
        # Get only cells inside this specific active row
        cells = row.find_elements(By.CLASS_NAME, "active-text")
        if cells:
            row_text = [cell.text.strip() for cell in cells]
            # Double check that we aren't pulling empty layout rows
            if any(row_text):
                csv_writer.writerow(row_text)
                count += 1
    return count

def get_current_first_row_index(driver):
    try:
        first_row = driver.find_element(By.CSS_SELECTOR, "tbody.ui-datatable-data tr[data-ri]")
        return first_row.get_attribute("data-ri")
    except:
        return None

def scrape_cidb_direct():
    file_exists = os.path.isfile(OUTPUT_FILE)
    csv_file = open(OUTPUT_FILE, mode='a', newline='', encoding='utf-8')
    writer = csv.writer(csv_file)
    if not file_exists:
        writer.writerow(HEADERS)

    driver = None
    page_number = 1
    max_pages = 675  # Updated to match the 675 count shown in your image!
    
    while page_number <= max_pages:
        if not driver:
            try:
                print("Launching fresh browser instance...")
                driver = setup_stealth_driver()
                driver.get(TARGET_URL)
                WebDriverWait(driver, 20).until(
                    EC.visibility_of_element_located((By.CLASS_NAME, "active-text"))
                )
            except Exception as e:
                print(f"Connection error: {e}. Retrying...")
                if driver: driver.quit()
                driver = None
                time.sleep(5)
                continue

        try:
            print(f"Processing Page {page_number}/{max_pages}...")
            
            # Keep track of current page's structural starting point
            initial_ri = get_current_first_row_index(driver)
            
            # Scrape active visible items only
            rows_saved = scrape_visible_rows(driver, writer)
            csv_file.flush()  
            print(f"Successfully saved exactly {rows_saved} rows from page {page_number}.")

            if page_number == max_pages:
                print("🎉 Dataset fully cleared and saved!")
                break

            # Target the next arrow button
            next_button = WebDriverWait(driver, 15).until(
                EC.presence_of_element_located((By.CLASS_NAME, "ui-paginator-next"))
            )
            
            if "ui-state-disabled" in next_button.get_attribute("class"):
                print("Reached end of pagination stream early.")
                break

            # Advance page
            driver.execute_script("arguments[0].scrollIntoView();", next_button)
            driver.execute_script("arguments[0].click();", next_button)
            
            # Block execution until the page values explicitly shift
            WebDriverWait(driver, 15).until(
                lambda d: get_current_first_row_index(d) != initial_ri
            )
            
            page_number += 1
            time.sleep(random.uniform(2.0, 3.5)) 

            # Memory optimization
            if page_number % 50 == 0:
                print("\n--- Clearing driver memory footprint ---")
                driver.quit()
                driver = None
                time.sleep(2)

        except Exception as e:
            print(f"Page {page_number} stalled or errored ({e}). Cycling window container...")
            if driver: driver.quit()
            driver = None  
            time.sleep(4)

    if driver: driver.quit()
    csv_file.close()
    print(f"\n Clean data extraction completed. Output file: '{OUTPUT_FILE}'")

if __name__ == "__main__":
    scrape_cidb_direct()

Launching fresh browser instance...
Processing Page 1/675...
Successfully saved exactly 10 rows from page 1.
Processing Page 2/675...
Successfully saved exactly 10 rows from page 2.
Processing Page 3/675...
Successfully saved exactly 10 rows from page 3.
Processing Page 4/675...
Successfully saved exactly 10 rows from page 4.
Processing Page 5/675...
Successfully saved exactly 10 rows from page 5.
Processing Page 6/675...
Successfully saved exactly 10 rows from page 6.
Processing Page 7/675...
Successfully saved exactly 10 rows from page 7.
Processing Page 8/675...
Successfully saved exactly 10 rows from page 8.
Processing Page 9/675...
Successfully saved exactly 10 rows from page 9.
Processing Page 10/675...
Successfully saved exactly 10 rows from page 10.
Processing Page 11/675...
Successfully saved exactly 10 rows from page 11.
Processing Page 12/675...
Successfully saved exactly 10 rows from page 12.
Processing Page 13/675...
Successfully saved exactly 10 rows from page 13.
Process